# Advanced Capabilities

Advanced geoms, stats, scales, positions, graph layouts, transitions, and interactive aesthetics.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo = Path.cwd()
if not (repo / 'ggplotly').exists():
    repo = next(parent for parent in Path.cwd().parents if (parent / 'ggplotly').exists())
sys.path.insert(0, str(repo))

from ggplotly import *

rng = np.random.default_rng(2026)

## Interval Geoms

In [ ]:
intervals = pd.DataFrame({
    'x': [1, 2, 3, 4, 5],
    'estimate': [8.0, 10.5, 9.2, 12.4, 11.1],
    'low': [6.8, 9.1, 7.9, 10.8, 9.8],
    'high': [9.1, 12.0, 10.7, 14.1, 12.5]
})

(ggplot(intervals, aes(x='x', y='estimate', ymin='low', ymax='high'))
 + geom_crossbar(width=0.55, fill='rgba(31,119,180,0.22)', color='black')
 + geom_pointrange(color='black', size=2))

In [ ]:
horizontal = intervals.assign(label=['North', 'South', 'East', 'West', 'Central'])

(ggplot(horizontal, aes(x='estimate', y='label', xmin='low', xmax='high'))
 + geom_errorbarh(color='steelblue', height=6))

## Binning And Density Stats

In [ ]:
cloud = pd.DataFrame({
    'x': np.r_[rng.normal(-1.2, 0.55, 180), rng.normal(1.1, 0.7, 180)],
    'y': np.r_[rng.normal(0.2, 0.65, 180), rng.normal(1.4, 0.55, 180)]
})

(ggplot(cloud, aes(x='x', y='y'))
 + geom_bin2d(bins=16)
 + scale_fill_binned(low='#f7fbff', high='#08519c', name='count'))

In [ ]:
(ggplot(cloud, aes(x='x', y='y'))
 + geom_hex(bins=14, palette='Viridis'))

In [ ]:
dot_data = pd.DataFrame({'x': np.round(rng.normal(0, 1, 120), 1)})
(ggplot(dot_data, aes(x='x')) + geom_dotplot(bins=24, dotsize=7, color='darkslateblue'))

In [ ]:
freq = pd.DataFrame({
    'x': np.r_[rng.normal(-0.6, 0.7, 160), rng.normal(0.8, 0.55, 160)],
    'group': ['baseline'] * 160 + ['new'] * 160
})

(ggplot(freq, aes(x='x', color='group'))
 + geom_freqpoly(bins=28, size=3)
 + scale_colour_manual({'baseline': '#1b9e77', 'new': '#d95f02'}))

## Distribution Extensions

In [ ]:
dist = pd.DataFrame({
    'group': ['A'] * 120 + ['B'] * 120 + ['C'] * 120,
    'value': np.r_[rng.normal(0.0, 0.7, 120), rng.normal(1.2, 0.5, 120), rng.normal(2.0, 0.9, 120)]
})

(ggplot(dist, aes(x='group', y='value'))
 + geom_slabinterval(fill='rgba(44,160,44,0.35)', color='#2ca02c'))

In [ ]:
(ggplot(dist, aes(x='value', y='group'))
 + geom_density_ridges(scale=0.75, fill='rgba(148,103,189,0.35)', color='#9467bd'))

In [ ]:
(ggplot(dist, aes(x='group', y='value'))
 + geom_beeswarm(width=0.55, size=6, alpha=0.65, color='teal'))

In [ ]:
(ggplot(dist, aes(x='group', y='value'))
 + geom_quasirandom(width=0.55, size=6, alpha=0.65, color='darkorange'))

## Label Placement And Text Paths

In [ ]:
labels = pd.DataFrame({
    'x': [1, 1.05, 1.1, 1.15, 2.0, 2.05, 2.1],
    'y': [1, 1.05, 0.95, 1.1, 2.0, 2.05, 1.95],
    'label': list('ABCDEFG')
})

(ggplot(labels, aes(x='x', y='y', label='label'))
 + geom_point(size=8, color='black')
 + geom_text_repel(force=0.14, color='crimson'))

In [ ]:
path = pd.DataFrame({
    'x': np.tile(np.linspace(0, 10, 80), 2),
    'y': np.r_[np.sin(np.linspace(0, 10, 80)), np.cos(np.linspace(0, 10, 80)) + 1.5],
    'group': np.repeat(['sin wave', 'cos wave'], 80),
    'label': np.repeat(['sin wave', 'cos wave'], 80)
})

(ggplot(path, aes(x='x', y='y', group='group', label='label'))
 + geom_labelpath(color='navy', fill='white', size=3))

## Categorical Flow, Mosaic, And Graphs

In [ ]:
flow = pd.DataFrame({
    'stage': ['Start', 'Middle', 'End'] * 5,
    'stratum': ['A', 'B', 'C', 'A', 'B', 'B', 'C', 'A', 'C', 'A', 'A', 'B', 'C', 'B', 'A'],
    'path': np.repeat(['p1', 'p2', 'p3', 'p4', 'p5'], 3)
})

(ggplot(flow, aes(x='stage', stratum='stratum', alluvium='path'))
 + geom_alluvium(alpha=0.45, size=12, color='rgba(31,119,180,0.35)'))

In [ ]:
mosaic = pd.DataFrame({
    'product': ['A', 'A', 'A', 'B', 'B', 'C', 'C', 'C', 'C'],
    'region': ['East', 'West', 'East', 'East', 'West', 'East', 'West', 'West', 'East']
})

(ggplot(mosaic, aes(x='product', fill='region')) + geom_mosaic())

In [ ]:
edges = pd.DataFrame({'source': ['A', 'A', 'B', 'C', 'D'], 'target': ['B', 'C', 'D', 'D', 'E']})
nodes = graph_layout(layout='circular').compute(edges, source='source', target='target')

(ggplot(edges, aes(**{'from': 'source', 'to': 'target'}))
 + geom_edge_link(layout=graph_layout(layout='circular'), color='rgba(80,80,80,0.45)')
 + geom_node_point(nodes, aes(x='x', y='y', label='node'), size=16, color='tomato')
 + geom_node_text(nodes, aes(x='x', y='y', label='node'), color='white'))

## Scale, Pattern, And Transition Helpers

In [ ]:
bars = pd.DataFrame({'x': ['one', 'two', 'one', 'two'], 'y': [2, 3, 4, 1], 'group': ['A', 'A', 'B', 'B']})

(ggplot(bars, aes(x='x', y='y', fill='group'))
 + geom_col_pattern(position='dodge')
 + scale_pattern_manual({'A': '/', 'B': 'x'})
 + scale_fill_viridis_d())

In [ ]:
multi = pd.DataFrame({'x': [1, 2, 1, 2], 'y': [1, 2, 3, 4], 'y2': [4, 3, 2, 1], 'g1': ['A', 'B', 'A', 'B'], 'g2': ['C', 'D', 'C', 'D']})

(ggplot(multi, aes(x='x'))
 + geom_point(aes(y='y', color='g1'), size=12)
 + scale_color_manual({'A': 'red', 'B': 'blue'})
 + new_scale_color()
 + geom_point(aes(y='y2', color='g2'), size=8, shape='diamond')
 + scale_color_manual({'C': 'green', 'D': 'orange'}))

In [ ]:
animated = pd.DataFrame({
    'x': list(range(5)) * 3,
    'y': [1, 2, 3, 2, 4, 2, 3, 5, 3, 6, 3, 5, 6, 5, 7],
    'time': np.repeat([2024, 2025, 2026], 5),
    'label': np.repeat(['baseline', 'mid', 'target'], 5)
})

(ggplot(animated, aes(x='x', y='y', tooltip='label', data_id='time'))
 + geom_point(size=10, color='steelblue')
 + transition_time('time'))